# Supplementary table: quality control

Builds the QC workbook for the final objects: every library from Cell Ranger to the all-cells and fibroblast/decidual
objects, with the nuclei each step removed and why. Values are read from the run's own files (Cell Ranger outputs, the
ingest and QC logs, the per-sample objects) and from the final objects. The per-nucleus QC calls are recomputed from
the counts with the rules of `qc.py` and must reproduce the counts in the logs exactly.

Not included: the reason each cluster was removed at annotation (Çağrı's annotation). Those nuclei are counted and
listed, with the step only.

In [ ]:
# Parameters
OUT = "Supplementary_Table_QC.xlsx"  # a new file; a relative path is next to the notebook

NH = "/n/lab_storage/eddy_lab/users/nhilgert"
NS = "/n/netscratch/eddy_lab/Everyone/nhilgert"
DATA = "/n/lab_storage/eddy_lab/Lab/mckinley/data"

SAMPLE_SHEET = f"{NH}/xmens_scripts/preprocess/samples.tsv"
QC_CONFIG = f"{NH}/xmens_scripts/preprocess/record/config/config.yaml"
CELLRANGER = NS + "/2026_11_06_cellranger_count/{lane}"  # the re-run of 6 Nov 2026 from the FASTQs
CELLBENDER_LOG = NH + "/decidualization/xmens_inputs_regen/{sample}/{sample}_cellbender.log"  # the July 2026 run
QC_RUN = f"{NS}/2026_11_06_qc_regen"  # ingest and QC of record: logs/, raw/ (CellBender calls), qc/ (kept)

# The QC'd objects sent to Çağrı on 6 Nov 2026, and his final objects.
QC_OBJECTS = {"mouse": f"{NS}/2026_11_06_qc_regen_nick_andy_coords/qc_passed_mouse.h5ad",
              "human": f"{NS}/2026_11_06_qc_regen_nick_andy_coords/qc_passed_human.h5ad"}
ALL_CELLS = {"mouse": f"{DATA}/20261110_mouse_allcells_final.h5ad",
             "human": f"{DATA}/20261107_Human_allcells_final.h5ad"}
FIBDEC_TYPES = ["eFib", "dec"]
FIBDEC_OBJECTS = {"mouse": f"{DATA}/20261110_mouse_only_eFib_dec_final.h5ad",
                  "human": f"{DATA}/20261107_Human_fibdec_final_nick_pdt.h5ad"}

# The runs kept the login HOME, so each environment also imported from HOME/.local; read, never written.
USER_BASE = "/n/home12/nhilgert/.local"
ENVS = {"ingest, concat and this notebook (rsc_uv)": f"{NH}/.uv/envs/rsc_uv",
        "QC and DoubletDetection (cuda11.8_env)": f"{NH}/.conda/envs/cuda11.8_env",
        "CellBender (cellbender_022)": f"{NH}/.conda/envs/cellbender_022"}

In [ ]:
import datetime
import gzip
import json
import os
import re
import subprocess
from pathlib import Path

import anndata as ad
import h5py
import numpy as np
import pandas as pd
import scanpy as sc
import yaml
from anndata.io import read_elem
from scipy.stats import median_abs_deviation

assert OUT, "set OUT to a new file"
assert not Path(OUT).exists(), f"{OUT} exists; choose a new name"
SPECIES = ["mouse", "human"]


def read_obs(path):
    """obs of an .h5ad, without reading its matrices."""
    with h5py.File(path, "r") as f:
        return read_elem(f["obs"])


def library_of(sample):
    """The 10x library at the end of a sample name: mck12a in 1.5dpi_late_mck12a."""
    return sample.rsplit("_", 1)[1]


def grab(pattern, text, cast=int):
    """The groups of the first match of pattern in text, cast; stops if there is none."""
    match = re.search(pattern, text)
    assert match, f"no match for {pattern!r}"
    values = tuple(cast(group) for group in match.groups())
    return values if len(values) > 1 else values[0]

## Libraries and sample names

The sample sheet names each library as the pipeline does; the final objects rename the mouse timepoints (+0.5 dpi).
The two are matched by library.

In [ ]:
sheet = pd.read_csv(SAMPLE_SHEET, sep="\t", dtype=str)
sheet["library"] = sheet["lane"].str.replace("_", "", regex=False)

all_cells = {species: read_obs(path) for species, path in ALL_CELLS.items()}
object_name = {}
for obs in all_cells.values():
    for name, timepoint in obs[["sample", "timepoint"]].astype(str).drop_duplicates().itertuples(index=False):
        assert library_of(name) not in object_name, f"two samples for library {library_of(name)}"
        object_name[library_of(name)] = (name, timepoint)
sheet["object_sample"] = sheet["library"].map(lambda library: object_name[library][0])
sheet["object_timepoint"] = sheet["library"].map(lambda library: object_name[library][1])
sheet = sheet.sort_values("species", key=lambda s: s.map(SPECIES.index), kind="stable").reset_index(drop=True)
print(sheet[["species", "lane", "sample_id", "object_sample", "object_timepoint"]].to_string(index=False))

## Cell Ranger

Metrics from the re-run of 6 Nov 2026 (`metrics_summary.csv`), reference and settings from its command line and
`_invocation`. CellBender (July 2026) read the February 2026 matrices; each lane's re-run raw matrix must be the same
file, byte for byte.

In [ ]:
def metric(text):
    """A Cell Ranger metric as a number: 93.1% -> 93.1, 3,765 -> 3765."""
    text = text.replace(",", "")
    return float(text[:-1]) if text.endswith("%") else int(text)


def current_path(recorded):
    """The runs recorded /n/eddy_lab/...; the same folders are now under /n/lab_storage/eddy_lab/."""
    return Path(recorded.replace("/n/eddy_lab/", "/n/lab_storage/eddy_lab/"))


def reference_description(folder):
    """Genome and annotation of a Cell Ranger reference: reference.json and the GENCODE header of its GTF."""
    info = json.loads((folder / "reference.json").read_text())
    gtf = next(folder.glob("genes/genes.gtf*"))
    with (gzip.open(gtf, "rt") if gtf.suffix == ".gz" else open(gtf)) as handle:
        header = [line.lstrip("#!").strip() for line in (next(handle) for _ in range(10)) if line.startswith("#")]
    described = [line.split(":", 1)[1].strip() for line in header if line.startswith("description:")]
    mkref = re.search(r"(\d+\.\d+\.\d+)", str(info.get("mkref_version", "")))
    return {"genome": ", ".join(info["genomes"]), "annotation": described[0] if described else "",
            "mkref_version": mkref.group(1) if mkref else ""}


cellranger = {}
for lane in sheet["lane"]:
    folder = Path(CELLRANGER.format(lane=lane))
    metrics = pd.read_csv(folder / "outs" / "metrics_summary.csv", dtype=str).iloc[0]
    invocation = (folder / "_invocation").read_text()
    reference = current_path(grab(r"--transcriptome=(\S+)", (folder / "_cmdline").read_text(), str))
    cellranger[lane] = {
        "metrics": {name: metric(value) for name, value in metrics.items()},
        "version": json.loads((folder / "_versions").read_text())["pipelines"].removeprefix("cellranger-"),
        "chemistry": grab(r"\bchemistry\s*=\s*\"([^\"]+)\"", invocation, str),
        "include_introns": grab(r"\binclude_introns\s*=\s*(\w+)", invocation, str),
        "reference": reference.name, **reference_description(reference)}


def identical(a, b, block=1 << 24):
    """True if two files hold the same bytes."""
    if os.path.getsize(a) != os.path.getsize(b):
        return False
    with open(a, "rb") as first, open(b, "rb") as second:
        while True:
            x, y = first.read(block), second.read(block)
            if x != y:
                return False
            if not x:
                return True


def placeholders(command):
    """A Cell Ranger command line with its per-lane values replaced, so the lanes can be compared."""
    for flag in ("id", "transcriptome", "fastqs", "sample", "output-dir"):
        command = re.sub(rf"--{flag}=\S+", f"--{flag}=<{flag}>", command)
    return command.split(" ", 1)[1].strip()


cellranger_commands = set()
for sample, lane in sheet[["sample_id", "lane"]].itertuples(index=False):
    folder = Path(CELLRANGER.format(lane=lane))
    cellranger_commands.add(placeholders((folder / "_cmdline").read_text()))
    read_by_cellbender = current_path(grab(r"--input (\S+)", Path(CELLBENDER_LOG.format(sample=sample)).read_text(),
                                           str))
    assert identical(folder / "outs" / "raw_feature_bc_matrix.h5", read_by_cellbender), f"{lane}: matrices differ"
assert len(cellranger_commands) == 1, cellranger_commands
print("raw matrices identical to those CellBender read: all", len(sheet), "lanes")
print(pd.DataFrame({lane: {k: v for k, v in run.items() if k != "metrics"} for lane, run in cellranger.items()}).T)

## Ingest and QC logs

One ingest log and one QC log per sample. Within each sample the counts must add up.

In [ ]:
qc_log, ingest_log = {}, {}
for sample in sheet["sample_id"]:
    text = Path(f"{QC_RUN}/logs/01_qc_{sample}.log").read_text()
    entry = {
        "called": grab(r"loaded (\d+) cells", text),
        "mito_genes": grab(r"flagged (\d+) mito genes", text),
        "umi_floor": grab(r"min \d+ counts: removed (\d+)", text),
        "mito": grab(r"filter pct_counts_mt: removed (\d+)", text),
        "depth_mad": grab(r"filter total_counts: removed (\d+)", text),
        "genes_mad": grab(r"filter n_genes_by_counts: removed (\d+)", text),
        "after_thresholds": grab(r"after threshold QC: (\d+)/", text),
        "doublets": grab(r"flagged (\d+) doublets", text),
        "after_qc": grab(r"final: (\d+) cells", text),
        "umis_bounds": grab(r"MAD total_counts \(log10\): bounds=\[([\d.]+), ([\d.]+)\]", text, float),
        "genes_bounds": grab(r"MAD n_genes_by_counts \(log10\): bounds=\[([\d.]+), ([\d.]+)\]", text, float),
        "doublet_settings": grab(r"(DoubletDetection [^\n]+)", text, str)}
    removed = entry["umi_floor"] + entry["mito"] + entry["depth_mad"] + entry["genes_mad"]
    assert entry["called"] - removed == entry["after_thresholds"], sample
    assert entry["after_thresholds"] - entry["doublets"] == entry["after_qc"], sample
    qc_log[sample] = entry

    text = Path(f"{QC_RUN}/logs/00b_ingest_{sample}.log").read_text()
    ingest_log[sample] = {
        "called": grab(r"loaded (\d+) cells", text),
        "with_coords_row": grab(r"barcode overlap: (\d+)/", text),
        "positioned": grab(r"placed cells \(non-null dbscan x AND y\): (\d+)/", text),
        "coords": grab(r"reading coords: (\S+) \(([^)]+)\)", text, str)}
    assert ingest_log[sample]["called"] == entry["called"], sample
print(pd.DataFrame(qc_log).T.drop(columns=["doublet_settings"]).to_string())

## Every nucleus removed at QC, recomputed from the counts

The rules of `qc.py`: nuclei below the UMI floor go first; on the rest, flags for the mitochondrial fraction and for
log10(x + 1) of total UMIs and of detected genes beyond the MAD bounds (raw MAD, unscaled), all computed once; the
flags are applied in the order mitochondrial, UMIs, genes, and a nucleus is counted under the first that removes it.
Doublets are the nuclei that passed the thresholds but are missing from the sample's QC output.

In [ ]:
config = yaml.safe_load(Path(QC_CONFIG).read_text())
MIN_COUNTS, N_MADS, MAX_PCT_MITO = (config["qc"][key] for key in ("min_counts", "n_mads", "max_pct_mito"))
FILTERS = [("pct_counts_mt", "mitochondrial fraction"), ("total_counts", "count-depth MAD"),
           ("n_genes_by_counts", "detected-gene MAD")]


def log10p(x):
    return np.log10(x + 1)


def outlier_bounds(values, n_mads):
    """As qc.py: the median +/- n_mads raw MADs."""
    median = np.nanmedian(values)
    mad = median_abs_deviation(values, nan_policy="omit")
    return median - n_mads * mad, median + n_mads * mad


qc_rows, cutoffs, positioned_called = [], {}, {}
for sample, species, library, name in sheet[
        ["sample_id", "species", "library", "object_sample"]].itertuples(index=False):
    raw = ad.read_h5ad(f"{QC_RUN}/raw/{sample}.h5ad")
    raw.var["mt"] = raw.var_names.str.startswith(("mt-", "MT-"))
    assert int(raw.var["mt"].sum()) == qc_log[sample]["mito_genes"], sample
    sc.pp.calculate_qc_metrics(raw, qc_vars=["mt"], percent_top=None, log1p=False, inplace=True)
    obs = raw.obs
    positioned_called[sample] = int(obs["x_um_dbscan"].notna().sum())

    below = obs["total_counts"] < MIN_COUNTS
    above = obs.loc[~below]
    flags, limits = {"pct_counts_mt": (above["pct_counts_mt"] > MAX_PCT_MITO).to_numpy()}, {}
    for column in ("total_counts", "n_genes_by_counts"):
        values = log10p(above[column].to_numpy())
        low, high = outlier_bounds(values, N_MADS)
        flags[column] = (values < low) | (values > high)
        limits[column] = (float(10 ** low - 1), float(10 ** high - 1))
    cutoffs[sample] = limits

    step = pd.Series("", index=obs.index, dtype=object)
    step[below.to_numpy()] = "UMI floor"
    remaining = np.ones(len(above), dtype=bool)
    for column, label in FILTERS:
        step[above.index[remaining & flags[column]]] = label
        remaining &= ~flags[column]
    passed = above.index[remaining]
    kept = read_obs(f"{QC_RUN}/qc/{sample}.h5ad").index
    assert kept.isin(passed).all(), f"{sample}: kept nuclei that failed a threshold"
    step[passed.difference(kept)] = "doublet"

    # The recomputed calls must give the log's counts, and the log's bounds to its one decimal.
    counts = step.value_counts()
    log = qc_log[sample]
    for label, key in (("UMI floor", "umi_floor"), ("mitochondrial fraction", "mito"),
                       ("count-depth MAD", "depth_mad"), ("detected-gene MAD", "genes_mad"), ("doublet", "doublets")):
        assert int(counts.get(label, 0)) == log[key], f"{sample} {label}: {counts.get(label, 0)} vs log {log[key]}"
    for column, key in (("total_counts", "umis_bounds"), ("n_genes_by_counts", "genes_bounds")):
        assert np.allclose(np.round(limits[column], 1), log[key]), f"{sample} {column} bounds"

    gone = step[step != ""]
    lows = {column: limits[column][0] for column in limits}
    highs = {column: limits[column][1] for column in limits}
    for barcode, label in gone.items():
        umis, genes, mito = (obs.at[barcode, c] for c in ("total_counts", "n_genes_by_counts", "pct_counts_mt"))
        reason = {
            "UMI floor": f"total UMIs {umis:.0f} < {MIN_COUNTS}",
            "mitochondrial fraction": f"mitochondrial UMIs {mito:.2f}% > {MAX_PCT_MITO}%",
            "count-depth MAD": f"total UMIs {umis:.0f} outside "
                               f"[{lows['total_counts']:.1f}, {highs['total_counts']:.1f}]",
            "detected-gene MAD": f"detected genes {genes:.0f} outside "
                                 f"[{lows['n_genes_by_counts']:.1f}, {highs['n_genes_by_counts']:.1f}]",
            "doublet": "predicted doublet (DoubletDetection)"}[label]
        qc_rows.append({"Species": species, "Sample": name, "Library": library, "Nucleus": f"{barcode}-{sample}",
                        "Cell barcode": barcode.split("-")[0], "Stage": "QC", "Step": label, "Reason": reason,
                        "Total UMIs": int(umis), "Detected genes": int(genes),
                        "Mitochondrial %": round(float(mito), 3)})
    assert positioned_called[sample] == ingest_log[sample]["positioned"], sample
    print(f"{sample}: {len(gone):,} removed, as in the log")

## The QC'd objects and the final objects

The QC'd objects hold, per sample, exactly the nuclei the QC kept. The all-cells objects are subsets of them; the
fibroblast/decidual nuclei are those of types eFib and dec, and must be the nuclei of the fibroblast/decidual objects.

In [ ]:
qc_objects = {species: read_obs(path) for species, path in QC_OBJECTS.items()}
fibdec = {}
for species in SPECIES:
    qc_obs, cells = qc_objects[species], all_cells[species]
    for sample in sheet.loc[sheet["species"] == species, "sample_id"]:
        expected = read_obs(f"{QC_RUN}/qc/{sample}.h5ad").index + f"-{sample}"
        in_object = qc_obs.index[qc_obs["sample"].astype(str) == sample]
        assert expected.sort_values().equals(in_object.sort_values()), f"{sample}: QC'd object differs from the QC"
    assert cells.index.isin(qc_obs.index).all(), f"{species}: all-cells nuclei missing from the QC'd object"
    fibdec[species] = cells.index[cells["cell_type"].astype(str).isin(FIBDEC_TYPES).to_numpy()]
    in_file = read_obs(FIBDEC_OBJECTS[species]).index
    assert fibdec[species].sort_values().equals(in_file.sort_values()), f"{species}: fib/dec object differs"
    print(f"{species}: {len(qc_obs):,} QC'd, {len(cells):,} all-cells, {len(fibdec[species]):,} fib/dec nuclei")

annotation_rows = []
for species in SPECIES:
    qc_obs, cells = qc_objects[species], all_cells[species]
    dropped = qc_obs.loc[~qc_obs.index.isin(cells.index)]
    names = dict(zip(sheet["sample_id"], sheet["object_sample"]))
    for nucleus, row in dropped.iterrows():
        annotation_rows.append({
            "Species": species, "Sample": names[str(row["sample"])], "Library": library_of(str(row["sample"])),
            "Nucleus": nucleus, "Cell barcode": nucleus.split("-")[0], "Stage": "annotation",
            "Step": "not in the all-cells object", "Reason": "", "Total UMIs": int(row["total_counts"]),
            "Detected genes": int(row["n_genes_by_counts"]), "Mitochondrial %": round(float(row["pct_counts_mt"]), 3)})
removed_barcodes = pd.DataFrame(qc_rows + annotation_rows)
print(removed_barcodes.groupby(["Species", "Stage", "Step"]).size().to_string())

## Nuclei per step

One row per sample, from the CellBender calls to the final objects; the columns "with a position" count and remove
nothing. Each row must add up.

In [ ]:
rows = []
for sample, species, library, name, timepoint in sheet[
        ["sample_id", "species", "library", "object_sample", "object_timepoint"]].itertuples(index=False):
    log = qc_log[sample]
    qc_obs = qc_objects[species]
    in_qc = qc_obs.loc[qc_obs["sample"].astype(str) == sample]
    cells = all_cells[species]
    in_cells = cells.loc[cells["sample"].astype(str) == name]
    in_fibdec = in_cells.loc[in_cells.index.isin(fibdec[species])]
    row = {"Species": species, "Timepoint": timepoint, "Sample": name, "Library": library,
           "CellBender-called nuclei": log["called"],
           "Called nuclei with a Slide-tags position": positioned_called[sample],
           "Removed: UMI floor": log["umi_floor"], "Removed: mitochondrial fraction": log["mito"],
           "Removed: count-depth MAD": log["depth_mad"], "Removed: detected-gene MAD": log["genes_mad"],
           "Removed: doublet": log["doublets"], "Nuclei after QC": len(in_qc),
           "Post-QC nuclei with a Slide-tags position": int(in_qc["x_um_dbscan"].notna().sum()),
           "Removed at annotation": len(in_qc) - len(in_cells), "Nuclei in the all-cells object": len(in_cells),
           "All-cells nuclei with a Slide-tags position": int(in_cells["x_um_dbscan"].notna().sum()),
           "Not in the fibroblast/decidual object": len(in_cells) - len(in_fibdec),
           "Nuclei in the fibroblast/decidual object": len(in_fibdec),
           "Fibroblast/decidual nuclei with a Slide-tags position": int(in_fibdec["x_um_dbscan"].notna().sum())}
    removed_at_qc = sum(row[f"Removed: {step}"] for step in ("UMI floor", "mitochondrial fraction", "count-depth MAD",
                                                             "detected-gene MAD", "doublet"))
    assert row["CellBender-called nuclei"] - removed_at_qc == row["Nuclei after QC"] == log["after_qc"], sample
    assert row["Nuclei after QC"] - row["Removed at annotation"] == row["Nuclei in the all-cells object"], sample
    rows.append(row)
waterfall = pd.DataFrame(rows)
totals = waterfall.groupby("Species", sort=False).sum(numeric_only=True).reset_index()
totals["Timepoint"], totals["Sample"], totals["Library"] = "all", totals["Species"] + " total", ""
waterfall = pd.concat([waterfall, totals[waterfall.columns]], ignore_index=True)
for species in SPECIES:
    total = waterfall.loc[waterfall["Sample"] == f"{species} total"].iloc[0]
    assert total["Nuclei after QC"] == len(qc_objects[species])
    assert total["Nuclei in the all-cells object"] == len(all_cells[species])
    assert total["Nuclei in the fibroblast/decidual object"] == len(fibdec[species])
print(waterfall.loc[waterfall["Timepoint"] == "all"].T.to_string())

## Samples and post-QC metrics

In [ ]:
CELLRANGER_COLUMNS = ["Number of Reads", "Estimated Number of Cells", "Mean Reads per Cell", "Median Genes per Cell",
                      "Median UMI Counts per Cell", "Valid Barcodes", "Sequencing Saturation", "Q30 Bases in Barcode",
                      "Q30 Bases in RNA Read", "Q30 Bases in UMI", "Reads Mapped to Genome",
                      "Reads Mapped Confidently to Genome", "Reads Mapped Confidently to Intronic Regions",
                      "Reads Mapped Confidently to Exonic Regions", "Reads Mapped Confidently to Transcriptome",
                      "Fraction Reads in Cells"]
samples = []
for sample, species, lane, library, name, timepoint, sheet_timepoint in sheet[
        ["sample_id", "species", "lane", "library", "object_sample", "object_timepoint", "timepoint"]
        ].itertuples(index=False):
    run, log, ingest = cellranger[lane], qc_log[sample], ingest_log[sample]
    row = {"Species": species, "Library": library, "Cell Ranger run": lane, "Pipeline sample": sample,
           "Sample (objects)": name, "Timepoint or phase (objects)": timepoint,
           "Timepoint (sample sheet)": sheet_timepoint, "Reference": run["reference"],
           "Genome": run["genome"], "Annotation": run["annotation"], "Cell Ranger mkref": run["mkref_version"],
           "Cell Ranger version": run["version"],
           "Chemistry setting": run["chemistry"], "Include introns": run["include_introns"]}
    for column in CELLRANGER_COLUMNS:
        unit = " (%)" if isinstance(run["metrics"][column], float) else ""
        row[f"Cell Ranger: {column[0].lower() + column[1:]}{unit}"] = run["metrics"][column]
    called = log["called"]
    row.update({
        "CellBender-called nuclei": called,
        "Reads per CellBender-called nucleus": round(run["metrics"]["Number of Reads"] / called),
        "QC: nuclei above the UMI floor": called - log["umi_floor"],
        "QC: total UMIs lower cutoff": round(cutoffs[sample]["total_counts"][0], 1),
        "QC: total UMIs upper cutoff": round(cutoffs[sample]["total_counts"][1], 1),
        "QC: detected genes lower cutoff": round(cutoffs[sample]["n_genes_by_counts"][0], 1),
        "QC: detected genes upper cutoff": round(cutoffs[sample]["n_genes_by_counts"][1], 1),
        "Slide-tags: called nuclei with a coordinates row": ingest["with_coords_row"],
        "Slide-tags: called nuclei with a position": ingest["positioned"],
        "Slide-tags: positioned (% of called)": round(100 * ingest["positioned"] / called, 1),
        "Nuclei after QC": log["after_qc"],
        "Nuclei in the all-cells object": int((all_cells[species]["sample"].astype(str) == name).sum())})
    samples.append(row)
samples = pd.DataFrame(samples)

# Per-sample metrics of the all-cells nuclei, from the QC metrics stored with them.
post_qc = []
for species in SPECIES:
    cells = all_cells[species]
    metrics = qc_objects[species].loc[cells.index, ["total_counts", "n_genes_by_counts", "pct_counts_mt"]]
    for name, timepoint in sheet.loc[sheet["species"] == species, ["object_sample", "object_timepoint"]].itertuples(
            index=False):
        mine = (cells["sample"].astype(str) == name).to_numpy()
        values = metrics.loc[mine]
        post_qc.append({"Species": species, "Timepoint": timepoint, "Sample": name, "Library": library_of(name),
                        "Nuclei": int(mine.sum()),
                        "Median UMIs": values["total_counts"].median(), "Mean UMIs": values["total_counts"].mean(),
                        "Median detected genes": values["n_genes_by_counts"].median(),
                        "Mean detected genes": values["n_genes_by_counts"].mean(),
                        "Median mitochondrial %": values["pct_counts_mt"].median(),
                        "Mean mitochondrial %": values["pct_counts_mt"].mean(),
                        "Nuclei with a Slide-tags position": int(cells.loc[mine, "x_um_dbscan"].notna().sum())})
post_qc = pd.DataFrame(post_qc).round(2)
print(samples.T.iloc[:, :2].to_string())

## Parameters and versions

In [ ]:
def command_of(log_path):
    """CellBender's command line from its log, with the per-sample paths replaced by placeholders."""
    lines = Path(log_path).read_text().splitlines()
    command = lines[lines.index(next(l for l in lines if l.endswith("Command:"))) + 1]
    command = re.sub(r"--input \S+", "--input <raw_feature_bc_matrix.h5>", command)
    return re.sub(r"--output \S+", "--output <sample>/<sample>", command)


cellbender_commands = {command_of(CELLBENDER_LOG.format(sample=s)) for s in sheet["sample_id"]}
assert len(cellbender_commands) == 1, cellbender_commands
doublet_settings = {log["doublet_settings"] for log in qc_log.values()}
assert len(doublet_settings) == 1, doublet_settings
references = samples.groupby("Species")["Reference"].unique().map(", ".join)

parameters = pd.DataFrame([
    ("Alignment and counting", "Cell Ranger count version", ", ".join(samples["Cell Ranger version"].unique()), ""),
    ("Alignment and counting", "Reference (mouse)", references["mouse"], ""),
    ("Alignment and counting", "Reference (human)", references["human"], ""),
    ("Alignment and counting", "Chemistry", ", ".join(samples["Chemistry setting"].unique()), ""),
    ("Alignment and counting", "Include introns", ", ".join(samples["Include introns"].unique()), ""),
    ("Alignment and counting", "Cell Ranger command", next(iter(cellranger_commands)),
     "the 6 Nov 2026 re-run; the same for every lane"),
    ("Ambient RNA removal", "CellBender command", cellbender_commands.pop(), "the same for every library"),
    ("QC", "UMI floor", MIN_COUNTS, "nuclei with fewer total UMIs removed first"),
    ("QC", "Mitochondrial fraction", MAX_PCT_MITO, "% of UMIs, genes named mt- (mouse) or MT- (human)"),
    ("QC", "MAD bounds", N_MADS, "raw MADs of log10(x + 1) total UMIs and detected genes, per sample, "
                                 "on the nuclei above the UMI floor"),
    ("QC", "Filter order", "UMI floor, mitochondrial, count depth, detected genes, doublets",
     "a nucleus is counted under the first step that removes it"),
    ("Doublets", "DoubletDetection", doublet_settings.pop(), "as logged"),
    ("Doublets", "p-value threshold", config["doublet"]["p_thresh"], ""),
    ("Doublets", "Voter threshold", config["doublet"]["voter_thresh"], ""),
], columns=["Stage", "Parameter", "Value", "Note"])


VERSION_CHECK = """
import datetime, importlib, os, site, sys
DIST = {"umap": "umap-learn", "sklearn": "scikit-learn"}
user = site.getusersitepackages() if site.ENABLE_USER_SITE else ""
print("python", sys.version.split()[0], "", "environment")
for name in sys.argv[1:]:
    try:
        module = importlib.import_module(name)
    except Exception:
        continue
    version = getattr(module, "__version__", "")
    if not version:
        import pkg_resources
        version = pkg_resources.get_distribution(DIST.get(name, name)).version
    where = "user site (HOME/.local)" if user and module.__file__.startswith(user) else "environment"
    date = datetime.date.fromtimestamp(os.path.getmtime(module.__file__)).isoformat()
    print(name, version, date, where)
"""
# The packages each environment was used for, as that environment imports them: dist-info folders can be stale
# (cuda11.8_env holds two for anndata), and HOME/.local comes first on the path. Dates are those of the imported files.
USED = {"ingest, concat and this notebook (rsc_uv)": ["scanpy", "anndata", "numpy", "scipy", "pandas", "h5py",
                                                      "openpyxl"],
        "QC and DoubletDetection (cuda11.8_env)": ["scanpy", "anndata", "doubletdetection", "numpy", "scipy",
                                                   "pandas", "sklearn", "leidenalg", "igraph", "umap", "pynndescent",
                                                   "numba", "joblib", "threadpoolctl", "natsort"],
        "CellBender (cellbender_022)": ["cellbender", "torch", "numpy"]}
# The user site must be on the path: cuda11.8_env imports DoubletDetection and scanpy from HOME/.local.
RUN_ENV = dict(os.environ, PYTHONUSERBASE=USER_BASE, PYTHONDONTWRITEBYTECODE="1")
RUN_ENV.pop("PYTHONNOUSERSITE", None)

mkref = samples.groupby("Species")["Cell Ranger mkref"].unique().map(", ".join)
versions = [("Cell Ranger count", ", ".join(samples["Cell Ranger version"].unique()), "", "the Cell Ranger runs", ""),
            ("Cell Ranger mkref (mouse reference)", mkref["mouse"], "", "reference.json", ""),
            ("Cell Ranger mkref (human reference, 10x build)", mkref["human"], "", "reference.json", "")]
for label, env in ENVS.items():
    result = subprocess.run([f"{env}/bin/python", "-c", VERSION_CHECK, *USED[label]], capture_output=True, text=True,
                            check=True, timeout=900, env=RUN_ENV)
    found = []
    for line in result.stdout.splitlines():
        tool, version, date, where = line.split(" ", 3)
        versions.append((tool, version, date, label, where))
        found.append(tool)
    missing = [name for name in USED[label] if name not in found]
    assert not missing, f"{label}: not importable in its environment: {missing}"
versions = pd.DataFrame(versions, columns=["Tool", "Version", "Installed", "Environment", "Imported from"])
print(versions.to_string(index=False))

## README and the workbook

In [ ]:
DEFINITIONS = {
    "Samples": "One row per 10x library: names, Cell Ranger run and metrics, CellBender calls, QC cutoffs, positions.",
    "Nuclei_per_step": "Nuclei per sample at every step from the CellBender calls to the final objects.",
    "Removed_barcodes": "One row per nucleus removed between the CellBender calls and the all-cells objects, with the "
                        "step that removed it. Reasons at annotation are not recorded in the objects.",
    "Post_QC": "Per-sample metrics of the nuclei in the all-cells objects.",
    "Parameters": "Settings of each step, as configured or logged.",
    "Versions": "Software versions, with the date each package was installed in its environment.",
}
objects = "; ".join(
    f"{species}: QC'd {Path(QC_OBJECTS[species]).name} ({len(qc_objects[species]):,} nuclei), all-cells "
    f"{Path(ALL_CELLS[species]).name} ({len(all_cells[species]):,}), fibroblast/decidual "
    f"{Path(FIBDEC_OBJECTS[species]).name} ({len(fibdec[species]):,})" for species in SPECIES)
readme = pd.DataFrame(
    [("Supplementary table: quality control of the Slide-tags single-nucleus RNA-seq data (mouse and human), from "
      "the Cell Ranger runs to the final objects.", ""),
     (f"Objects described: {objects}.", ""),
     (f"Built {datetime.date.today().isoformat()} by tables/supplementary_table_qc.ipynb from the files the "
      "pipeline wrote; every per-nucleus QC call was recomputed and checked against the QC logs.", ""),
     ("Cell Ranger values are from the re-run of 6 Nov 2026 from the FASTQs; its raw matrices are byte-identical to "
      "the February 2026 matrices that CellBender read.", ""),
     ("", "")] + [(f"Sheet {name}", text) for name, text in DEFINITIONS.items()],
    columns=["Note", "Definition"])

sheets = {"README": readme, "Samples": samples, "Nuclei_per_step": waterfall, "Removed_barcodes": removed_barcodes,
          "Post_QC": post_qc, "Parameters": parameters, "Versions": versions}
with pd.ExcelWriter(OUT, engine="openpyxl") as writer:
    for name, table in sheets.items():
        table.to_excel(writer, sheet_name=name, index=False)
        sheet_out = writer.sheets[name]
        sheet_out.freeze_panes = "A2"
        for cells in sheet_out.columns:
            width = max(len(str(cell.value)) if cell.value is not None else 0 for cell in cells[:200])
            sheet_out.column_dimensions[cells[0].column_letter].width = min(max(10, width + 2), 60)
print(f"wrote {Path(OUT).name}: " + ", ".join(f"{name} ({len(table):,} rows)" for name, table in sheets.items()))